In [1]:
import findspark
findspark.init()
import pyspark
import os
import sys
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql import types as T
import matplotlib.pyplot as plt
plt.style.use(style='seaborn')

from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler

spark = SparkSession.builder.appName('Linear Regression').getOrCreate()
spark

#### User Define Function (UDF)

In [2]:
# Defining a function unpackListUDF to unpack list.
unpackListUDF = F.udf (
    lambda x: list(set([item for sublist in x for item in sublist])),
    T.ArrayType(T.StringType())
)

meanUDF = F.udf (
    lambda x: float(np.mean(x)),
    T.FloatType()
)

medianUDF = F.udf (
    lambda x: float(np.median(x)),
    T.FloatType()
)

In [ ]:
# Read the dataset
df = spark.read.csv('CaliforniaHousingWHeader.csv', header = True, inferSchema = True)
df.show()
df.printSchema()

In [ ]:
# The features longtitude and latitude are independent of the medianHouseValue. We can drop those two features
featureToDrop = ['longtitude','latitude','medianIncome']
df1 = df.select([c for c in df.columns if c not in featureToDrop])
df1.show()
df1.printSchema()

In [ ]:
# Check if there is any NULL values. If yes, remove them
from pyspark.sql.functions import col, count, isnan, when
df1.select([count(when(col(c).isNull(), c)).alias(c) for c in df1.columns]).show()

#### The result indicates there is no null value features

In [ ]:
# split the features and label. We want to determine the medianHouseValue, hence, medianHouseValue is the label.
# exclude it from other features.
features = df1.drop('medianHouseValue')

In [ ]:
# Assemble all features together using VectorAssembler
myAssembler = VectorAssembler(
    inputCols = features.columns,
    outputCol = 'features'
)

output = myAssembler.transform(df1).select('features','medianHouseValue')
output.show(truncate = False)

In [ ]:
# split the dataset into 70% training and 30% test
(train, test) = output.randomSplit([0.7,0.3])

train.show()
test.show()

#### Linear Regression

In [ ]:
from pyspark.ml.regression import LinearRegression
linReg = LinearRegression(featuresCol = 'features', labelCol = 'medianHouseValue')
linearModel = linReg.fit(train)

In [ ]:
print('Coefficients: ' + str(linearModel.coefficients))
print('\nIntercept: ' + str(linearModel.intercept))

In [ ]:
trainSummary = linearModel.summary
print('RMSE: %f' % trainSummary.rootMeanSquaredError)
print('\nr2: %f' % trainSummary.r2)

In [ ]:
from pyspark.sql.functions import abs
predictions = linearModel.transform(test)
x = ((predictions['medianHouseValue']-predictions['prediction'])/predictions['medianHouseValue']) * 100
predictions = predictions.withColumn('Accuracy', abs(x))
predictions.select('prediction', 'medianHouseValue', 'Accuracy', 'features').show(truncate = False)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
myPredEvaluator = RegressionEvaluator (
    predictionCol = 'prediction',
    labelCol = 'medianHouseValue',
    metricName = 'r2'
)

print('R Squared (R2) on test data = %g' % myPredEvaluator.evaluate(predictions))

In [ ]:
def adjR2(x):
    r2 = trainSummary.r2
    n = df1.count()
    p = len(df1.columns)
    adjustedR2 = 1 - (1 - r2) * (n - 1) / (n-p-1)
    return adjustedR2

In [ ]:
adjR2(train)

In [ ]:
adjR2(test)

In [ ]:
linReg = LinearRegression(
    featuresCol = 'features', 
    labelCol = 'medianHouseValue',
    maxIter = 50,
    regParam = 0.12,
    elasticNetParam = 0.2)
linearModel = linReg.fit(train)

In [ ]:
linearModel.summary.rootMeanSquaredError

In [ ]:
featuresRDD = features.rdd.map(lambda row: row[0:])

In [ ]:
featuresRDD.collect()

In [ ]:
from pyspark.mllib.feature import StandardScaler
scaler1 = StandardScaler().fit(featuresRDD)
scaledFeatures = scaler1.transform(featuresRDD)

In [ ]:
for data in scaledFeatures.collect():
    print(data)

In [ ]:
df2 = scaledFeatures.map(lambda x: (x, )).toDF(['ScaledData'])

In [ ]:
df2.show(truncate = False)